In [1]:
import os
# Set this to "1" to disable torch.compile globally
# os.environ["TORCH_COMPILE_DISABLE"] = "1"
import torch
import torch.nn as nn

import numpy as np
from typing import NamedTuple
import matplotlib.pyplot as plt
from scipy.stats import gamma, poisson

from models.utils import build_warmup_epochs
from models.inn import RealNVP,RealNVPSummary,RealNVPSummarySingle
from models.regressionNetwork import RegressionNetwork, train_regression_network

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

from tqdm import tqdm
import torch.nn.utils as utils

In [2]:
dataset = np.load("small_dataset.npz")

seed = 67
print(dataset)
foot = dataset["foot"]

# pad leg with a 1 to make it 2 elemental
leg_pre = dataset["leg"]
leg = np.random.normal(0,1,(leg_pre.shape[0],2))
leg[:,0]= leg_pre.flatten()

# flatten foot
foot = foot.reshape((foot.shape[0],-1)) # flatten(1)?

NpzFile 'small_dataset.npz' with keys: foot, leg


In [3]:
dataset = np.load("big_dataset.npz") #TODO: what are we training here///!

seed = 67
print(dataset)
foot = dataset["foot"]

# pad leg with a 1 to make it 2 elemental
leg = dataset["height"] * 0.45
print(f"{leg.shape=}")


# flatten foot
leg = leg.reshape((-1,1))
foot = foot.reshape((foot.shape[0],-1)) # flatten(1)?


NpzFile 'big_dataset.npz' with keys: foot, com, height
leg.shape=(100000,)


In [4]:
dataset = np.load("dataset.npz") #TODO: what are we training here///!
print(dataset)
seed = 67
print(dataset)
foot = dataset["foot"]

# pad leg with a 1 to make it 2 elemental
leg = dataset["height"] * 0.45
print(f"{leg.shape=}")


# flatten foot
leg = leg.reshape((-1,1))
foot = foot.reshape((foot.shape[0],-1)) # flatten(1)?


NpzFile 'dataset.npz' with keys: foot, com, height
NpzFile 'dataset.npz' with keys: foot, com, height
leg.shape=(100000,)


In [5]:


print(f"{foot.shape}")
print(f"{leg.shape}")

assert foot.shape[0] == leg.shape[0]
N = foot.shape[0]
indices = np.arange(N)

# 2. Shuffle the indices in-place using a generator for better control

rng = np.random.default_rng(seed)
rng.shuffle(indices)

# 3. Calculate the split point
split_point = int(N * 0.8)

# 4. Slice the shuffled array into two vectors
train_indices = indices[:split_point]
test_indices = indices[split_point:]

# declare final arrays
train_foot = foot[train_indices]
train_leg = leg[train_indices]


test_foot = foot[test_indices]
test_leg = leg[test_indices]
print(test_foot.shape,test_leg.shape)

(100000, 234)
(100000, 1)
(20000, 234) (20000, 1)


In [6]:
# create dataloaders:
from torch.utils.data import DataLoader,TensorDataset
dtype = torch.float32
train_tensor_leg = torch.tensor(train_leg).to(dtype)
train_tensor_foot = torch.tensor(train_foot).to(dtype)
test_tensor_leg = torch.tensor(test_leg).to(dtype)
test_tensor_foot = torch.tensor(test_foot).to(dtype)

train_foot_mean = train_tensor_foot.mean()
train_foot_std = train_tensor_foot.std()
train_leg_mean = train_tensor_leg.mean(0,keepdim=True)
train_leg_std = train_tensor_leg.std(0,keepdim=True)

print(train_foot_mean,train_foot_std,train_leg_mean,train_leg_std)

train_tensor_leg = (train_tensor_leg - train_leg_mean) / train_leg_std
train_tensor_foot = (train_tensor_foot - train_foot_mean) / train_foot_std
test_tensor_leg = (test_tensor_leg - train_leg_mean) / train_leg_std
test_tensor_foot = (test_tensor_foot - train_foot_mean) / train_foot_std


train_datatensor = TensorDataset(train_tensor_leg,train_tensor_foot)
test_datatensor = TensorDataset(test_tensor_leg,test_tensor_foot)

def create_dataloaders(batch_size,num_workers = 11):
    train_loader = DataLoader(
                train_datatensor,
                shuffle=True,
                batch_size=batch_size,
                num_workers=11,
                pin_memory=True,
                persistent_workers=True,
                drop_last=True # otherwise, some instances are weighed higher
            )

    test_loader = DataLoader(
                test_datatensor,
                shuffle=False, # unnecesarry
                batch_size=batch_size, # OPT: increase
                num_workers=max(num_workers // 2,4),
                pin_memory=True,
                persistent_workers=True,
                
            )
    return train_loader,test_loader


tensor(0.2109) tensor(0.7522) tensor([[0.7901]]) tensor([[0.0309]])


In [7]:
# print input sizes for model
train_loader = create_dataloaders(32)[0]
leg,foot = next(iter(train_loader))
print(foot.shape,leg.shape)


torch.Size([32, 234]) torch.Size([32, 1])


In [8]:
from torch import GradScaler
from models.inn import CouplingBlock
@torch.compile()
def calculate_loss(model: RealNVPSummarySingle, output):
    loss = torch.zeros((), requires_grad=True, device=device)
    # for block in model.realNVP.blocks: # use for summary network
    for block in model.realNVP.blocks:
        block: CouplingBlock
        loss = loss - block.block_det
    loss += output.pow(2).sum() / 2  # sum over all axis
    return loss / output.size(0)  # mean over batch


def train_inn_cond(
    model: RealNVP,
    train_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    optim: torch.optim.Optimizer,
    scaler: GradScaler,
    lr_scheduler: torch.optim.lr_scheduler.LRScheduler,
    epochs: int,
    history,
    batch_size=128,
):
    """
    model: A inn
    train_set_fn: a function that returns X,Y
    with:
    X: the hidden parameter that we want a posterior for later.
    Y: the condition (like observations that will be available at inference time)

    # scaler = GradScaler("cuda", enabled=(device.type == 'cuda'))
    """
    history["train_loss"] = history.get("train_loss", [])
    history["test_loss"] = history.get("test_loss", [])
    model.train()
    pbar = tqdm(range(epochs), desc="Training",leave=True)
    for epoch in pbar:
        train_epoch_loss = torch.zeros((1,), requires_grad=False, device=device)
        test_epoch_loss = torch.zeros((1,), requires_grad=False, device=device)
        train_epoch_length = len(train_loader)
        test_epoch_length = len(test_loader)
        # for now, define a epoch as 100 iterations, because
        # we have no train set size yet.
        
        for leg, foot in train_loader:
            leg: torch.Tensor
            foot: torch.Tensor
            X, Y = leg.to(device), foot.to(device)  # TODO: nonblocking
            optim.zero_grad()
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                #print(X.shape,Y.shape)
                output = model.forward(X, Y)
                # y is used for condition, x is input (see docstring)
                loss = calculate_loss(model, output)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            scaler.step(optim)
            scaler.update()
            train_epoch_loss += loss.mean().detach()

        with torch.no_grad():
            model.eval()
            for leg, foot in test_loader:
                leg: torch.Tensor
                foot: torch.Tensor
                X, Y = leg.to(device), foot.to(device)  # TODO: nonblocking
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    #print(X.shape,Y.shape)
                    output = model.forward(X, Y)
                    loss = calculate_loss(model, output)


                test_epoch_loss += loss.mean().detach()
            # print(loss.mean().item())

        model.train()
        train_avg_epoch_loss = (train_epoch_loss / train_epoch_length).item()
        test_avg_epoch_loss = (test_epoch_loss / test_epoch_length).item()
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_epoch_loss:.4f}", end="\r")
        pbar.set_postfix(train_loss=train_avg_epoch_loss,test_loss=test_avg_epoch_loss)
        history["train_loss"].append(train_avg_epoch_loss)
        history["test_loss"].append(test_avg_epoch_loss)
        lr_scheduler.step()
    return


In [9]:
# TODO: normalize inputs
input_size = 1  #
observable_size = 51 * 6  # maybe use the spacial structure of the data
reduced_obs_size = 16


# model = RealNVP(
#     input_size=input_size,
#     hidden_size=hidden_size,
#     blocks=num_coup_blocks,
#     condition_size=observable_size,
# )
model = RealNVPSummarySingle(
    input_size=input_size,
    condition_size=observable_size,
    reduced_condition_size=reduced_obs_size,
    s_hidden=256,
    s_layers=3,
    r_hidden=64,
    r_blocks=8,
)
# model = RealNVPSummary(input_size=input_size,condition_size=observable_size,reduced_condition_size=reduced_obs_size,s_hidden=256,s_layers=3,r_hidden=32,r_blocks=6)


model.to(device)
model = torch.compile(model, mode="max-autotune", fullgraph=True)

num_epochs = 40
lr_warmup_epochs = 10                      
init_lr = 1e-3
weight_decay = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), init_lr, weight_decay=weight_decay)
history = {}

batch_size = 32
train_loader, test_loader = create_dataloaders(batch_size)

scaler = torch.GradScaler("cuda", enabled=(device.type == "cuda"))
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, build_warmup_epochs(lr_warmup_epochs, num_epochs)
)


train_inn_cond(
    model,
    train_loader=train_loader,
    test_loader=test_loader,
    optim=optimizer,
    scaler=scaler,
    lr_scheduler=lr_scheduler,
    epochs=num_epochs,
    history=history,
    batch_size=None,
)
print(next(model.parameters()).device)

Training:   0%|          | 0/40 [00:00<?, ?it/s]E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0] failed while attempting to run meta for aten.mm.default
E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0] Traceback (most recent call last):
E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0]   File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/_subclasses/fake_tensor.py", line 2717, in _dispatch_impl
E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0]     r = func(*args, **kwargs)
E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0]         ^^^^^^^^^^^^^^^^^^^^^
E0330 00:41:28.699000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/0]   File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/_ops.py", line 829, in __call__
E0330 00:41:28.699000 1

TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in function linear>(*(FakeTensor(..., device='cuda:0', size=(32, 234)), Parameter(FakeTensor(..., device='cuda:0', size=(256, 306), requires_grad=True)), Parameter(FakeTensor(..., device='cuda:0', size=(256,), requires_grad=True))), **{}): got RuntimeError('a and b must have same reduction dim, but got [32, 234] X [306, 256].')

from user code:
   File "/home/david/dev/SBI_walking/models/inn.py", line 305, in forward
    hy = self.summary(y)
  File "/home/david/dev/SBI_walking/models/regressionNetwork.py", line 24, in forward
    return self.encoder(x)
  File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/nn/modules/container.py", line 244, in forward
    input = module(input)
  File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/nn/modules/linear.py", line 125, in forward
    return F.linear(input, self.weight, self.bias)

Set TORCHDYNAMO_VE

In [10]:
# TODO: normalize inputs
input_size = 1  #
observable_size = 51 * 6  # maybe use the spacial structure of the data
reduced_obs_size = 16


# model = RealNVP(
#     input_size=input_size,
#     hidden_size=hidden_size,
#     blocks=num_coup_blocks,
#     condition_size=observable_size,
# )
model = RealNVPSummarySingle(
    input_size=input_size,
    condition_size=observable_size,
    reduced_condition_size=reduced_obs_size,
    s_hidden=256,
    s_layers=3,
    r_hidden=64,
    r_blocks=8,
)
# model = RealNVPSummary(input_size=input_size,condition_size=observable_size,reduced_condition_size=reduced_obs_size,s_hidden=256,s_layers=3,r_hidden=32,r_blocks=6)


model.to(device)
model = torch.compile(model, mode="max-autotune", fullgraph=True)

num_epochs = 40
lr_warmup_epochs = 10                      
init_lr = 1e-3
weight_decay = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), init_lr, weight_decay=weight_decay)
history = {}

batch_size = 32
train_loader, test_loader = create_dataloaders(batch_size)

scaler = torch.GradScaler("cuda", enabled=(device.type == "cuda"))
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, build_warmup_epochs(lr_warmup_epochs, num_epochs)
)


train_inn_cond(
    model,
    train_loader=train_loader,
    test_loader=test_loader,
    optim=optimizer,
    scaler=scaler,
    lr_scheduler=lr_scheduler,
    epochs=num_epochs,
    history=history,
    batch_size=None,
)
print(next(model.parameters()).device)

Training:   0%|          | 0/40 [00:00<?, ?it/s]E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1] failed while attempting to run meta for aten.mm.default
E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1] Traceback (most recent call last):
E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1]   File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/_subclasses/fake_tensor.py", line 2717, in _dispatch_impl
E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1]     r = func(*args, **kwargs)
E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1]         ^^^^^^^^^^^^^^^^^^^^^
E0330 00:41:52.813000 13420 site-packages/torch/_subclasses/fake_tensor.py:2721] [0/1]   File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/_ops.py", line 829, in __call__
E0330 00:41:52.813000 1

TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in function linear>(*(FakeTensor(..., device='cuda:0', size=(32, 234)), Parameter(FakeTensor(..., device='cuda:0', size=(256, 306), requires_grad=True)), Parameter(FakeTensor(..., device='cuda:0', size=(256,), requires_grad=True))), **{}): got RuntimeError('a and b must have same reduction dim, but got [32, 234] X [306, 256].')

from user code:
   File "/home/david/dev/SBI_walking/models/inn.py", line 305, in forward
    hy = self.summary(y)
  File "/home/david/dev/SBI_walking/models/regressionNetwork.py", line 24, in forward
    return self.encoder(x)
  File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/nn/modules/container.py", line 244, in forward
    input = module(input)
  File "/home/david/miniconda3/envs/ml_homework/lib/python3.12/site-packages/torch/nn/modules/linear.py", line 125, in forward
    return F.linear(input, self.weight, self.bias)

Set TORCHDYNAMO_VE

In [ ]:
import matplotlib.pyplot as plt
model.to(device)
model.eval()
# 1. Set a clean style
plt.style.use('seaborn-v0_8-muted') # or 'ggplot'
plt.figure(figsize=(10, 5), dpi=100)

# 2. Plot with better aesthetics
plt.plot(np.array(history["train_loss"][:]), 
         color='#1f77b4',       # A nice professional blue
         linewidth=1,           # Slightly thicker line
         label='Training Loss')

plt.plot(np.array(history["test_loss"][:]), 
         color="#c50488",       # A nice professional blue
         linewidth=1,           # Slightly thicker line
         label='Test Loss')


# 3. Add context and labels
plt.title('Model Training Progress', fontsize=14, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)

# 4. Clean up the "frame"
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
#plt.yscale("log")
plt.show()

In [ ]:
# 1. Set a clean style
plt.style.use('seaborn-v0_8-muted') # or 'ggplot'
plt.figure(figsize=(10, 5), dpi=100)

# 2. Plot with better aesthetics
plt.plot(-np.array(history["test_loss"][len(history["test_loss"])//4 * 3: ]), 
         color='#1f77b4',       # A nice professional blue
         linewidth=2,           # Slightly thicker line
         label='test Loss')

# 3. Add context and labels
plt.title('Last forth of test loss', fontsize=14, pad=15)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)

# 4. Clean up the "frame"
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.yscale("log")
plt.show()

In [ ]:
# import torch

def compute_mmd(x, y, sigma=1.0):
    """
    Computes the Maximum Mean Discrepancy (MMD) between two sets of samples.
    Args:
        x: Tensor of shape [n, d] (Samples from Distribution P)
        y: Tensor of shape [m, d] (Samples from Distribution Q)
        sigma: Bandwidth of the RBF kernel
    """
    # 1. Compute distance matrix (n+m, n+m)
    z = torch.cat([x, y], dim=0)
    # Efficient pairwise distance calculation: ||a-b||^2 = ||a||^2 + ||b||^2 - 2<a,b>
    dist_matrix = torch.cdist(z, z, p=2)**2
    
    # 2. Apply Gaussian Kernel
    kernel_matrix = torch.exp(-dist_matrix / (2 * sigma**2))
    
    # 3. Extract sub-matrices
    n = x.size(0)
    m = y.size(0)
    
    k_xx = kernel_matrix[:n, :n]
    k_yy = kernel_matrix[n:, n:]
    k_xy = kernel_matrix[:n, n:]
    
    # 4. Compute MMD^2 (Unbiased estimator)
    # Subtracting the diagonal (self-distances) for k_xx and k_yy
    mmd2 = (k_xx.sum() - n) / (n * (n - 1)) + \
           (k_yy.sum() - m) / (m * (m - 1)) - \
           2 * k_xy.mean()
           
    return mmd2

# Example Usage
x = torch.randn(100, 2)  # Mean 0
y = torch.randn(100, 2) + 0.5  # Mean 0.5
loss = compute_mmd(x, y)
print(f"MMD: {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    latent = model.forward(train_tensor_leg.to(device),train_tensor_foot.to(device)).cpu()
    print(latent.shape)
    print(f"Train Set: {latent.std(0)=}")
    (print(f"{torch.cov(latent.T)=}"))


    plt.hist(latent[:,0],bins=50)
    plt.show()


In [ ]:

with torch.no_grad():
    latent = model.forward(test_tensor_leg.to(device),test_tensor_foot.to(device)).cpu()
    print(f"Test Set: {latent.std(0)=}")

    plt.hist(latent[:,0])
    plt.show()

In [ ]:
# visualize the dataset leg
plt.hist(leg_pre,bins=25)

In [ ]:
n_real = 1000

res  = model.reverse(z = torch.normal(0,1,(n_real,1)).to(device) ,y = test_tensor_foot[:n_real].to(device) ).detach()

In [ ]:
generated_corrected = res.cpu() * train_leg_std + train_leg_mean
generated_corrected = generated_corrected.detach()[:,0].flatten()

In [ ]:
plt.hist(generated_corrected,bins=25)


In [ ]:
plt.hist((leg_pre[:1000] - generated_corrected.numpy()).flatten() )

In [ ]:
n_real = 50
n_per_real = 4000


In [ ]:
# testing repeat interleave and reshape
a = torch.arange(6).reshape((3,2))
print(tmp:=a.repeat_interleave(3,dim=0))
tmp.reshape((3,3,2))

In [ ]:
truth_corrected_wrong_shape = test_tensor_leg[:n_real].cpu() * train_leg_std + train_leg_mean
truth_corrected = truth_corrected_wrong_shape.flatten()
sort_indices = truth_corrected.sort().indices
truth_corrected = truth_corrected[sort_indices]

In [ ]:
foot_data = test_tensor_foot[:n_real][sort_indices]
model.to(device)
res  = model.reverse(z = torch.normal(0,1,(n_real*n_per_real,1)).to(device) ,y = foot_data.repeat_interleave(n_per_real,dim=0).to(device) ).detach()
res = res.reshape((n_real,n_per_real,1))
print(res.shape)

generated_corrected_wrong_shape = res.flatten().cpu() * train_leg_std.flatten() + train_leg_mean.flatten()
generated_corrected = generated_corrected_wrong_shape.detach()



In [ ]:
plt.scatter(torch.arange(n_real).repeat_interleave(n_per_real).numpy(),generated_corrected,label="generated")
plt.scatter(np.arange(n_real) ,truth_corrected,label="truth")
plt.legend()

In [ ]:
# 1. Calculate statistics across the generated samples
# Reshape to (n_real, n_per_real) to easily compute stats per group
gen_reshaped = generated_corrected.reshape(n_real, n_per_real)
gen_mean = gen_reshaped.mean(axis=1)
gen_std = gen_reshaped.std(axis=1)

# 2. Define the x-axis
x = np.arange(n_real)

# 3. Plotting
plt.figure(figsize=(10, 6))

# Plot the confidence band (1 standard deviation)
plt.fill_between(x, gen_mean - gen_std, gen_mean + gen_std, 
                 color='tab:blue', alpha=0.2, label="Generated Confidence (1σ)")

# Plot the mean of the generated values
plt.plot(x, gen_mean, color='tab:blue', lw=2, label="Generated Mean")

# Plot the truth values
plt.scatter(x, truth_corrected.flatten(), color='tab:orange', s=20, label="Truth", zorder=3)

plt.xlabel("Real Index")
plt.ylabel("Value")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 1. Reshape the data to (n_real, n_per_real)
gen_reshaped = generated_corrected.reshape(n_real, n_per_real)

# 2. Calculate the 2.5th, 50th (median), and 97.5th percentiles
# axis=1 calculates the percentile across the multiple samples for each real value
lower_bound = np.percentile(gen_reshaped, 2.5, axis=1)
upper_bound = np.percentile(gen_reshaped, 97.5, axis=1)
median_gen = np.percentile(gen_reshaped, 50, axis=1)

# 3. Define the x-axis
x = np.arange(n_real)

# 4. Plotting
plt.figure(figsize=(12, 6))

# Plot the 95% confidence interval
plt.fill_between(x, lower_bound, upper_bound, 
                 color='tab:blue', alpha=0.2, label="95% Quantile Band")

# Plot the median (often more representative than the mean for skewed data)
plt.plot(x, median_gen, color='tab:blue', lw=2, label="Generated Median")

# Plot the truth values
plt.scatter(x, truth_corrected.flatten(), color='red', s=15, label="Truth", zorder=3)

plt.xlabel("Index")
plt.ylabel("LegLength")
plt.title("Generated Distribution vs. Truth (95% CI)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [11]:
filename_model_params = "saved_model_test.model"
filename_model_hyperparams = "saved_model_test.def"


In [ ]:
torch.save(model.state_dict(), filename_model_params)
model_hyperparams_dict = {
    "input_size": model.realNVP.input_size,
    "condition_size": model.summary.input_size,
    "reduced_condition_size": model.realNVP.condition_size,
    "s_hidden": model.summary.encoder_layers[0].weight.shape[0],
    "s_layers": model.summary.layers,
    "r_hidden": model.realNVP.blocks[0].scale_net[0].weight.shape[0],
    "r_blocks": len(model.realNVP.blocks),
}
torch.save(model_hyperparams_dict,filename_model_hyperparams)


In [ ]:
init_dict = torch.load(filename_model_hyperparams)
print(init_dict)
loaded_dict = torch.load(filename_model_params)
model = RealNVPSummarySingle(**init_dict)
model = torch.compile(model)
model: RealNVPSummarySingle
model.load_state_dict(loaded_dict)